In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Second-Order ODEs and Broadcasting in `numpy`

So far every ODE (ordinary differential equation) we've integrated has had a single number $y$ as its state.  But most of physics is Newton's second law, $F = ma$, which is a *second-order* ODE, because the acceleration $a$ is a *second derivative*,
\begin{align}
    a = \frac{d^2x}{dt^2}.
\end{align}
If we're interested in making predictions about where an object goes when its behavior is dictated by a second order equation we'll need to be able to solve differential equations of this type.
The forces we find in physics depend on time, position (think, for example, of gravity or springs) and velocity (friction, or air resistance).  So, the force $F$ is a function of $t$, $x$, and $v$ and therefore generically Newton's laws take the form
\begin{align}
    a &= \frac{1}{m} F(t, x, v)
\end{align}
and $v=dx/dt$ and $a=dv/dt = d^2x/dt^2$.  This is called the object's *equation of motion*.

In this lesson we'll see that
 - we can think of a second-order ODE as a group of two first-order ODEs,
 - the integrators for first-order ODEs we already know and love need only a small tweak to handle a state made of several numbers (think position and velocity)
 - numpy's rules for combining arrays of different shapes, called *broadcasting*, are already at work when we multiply a two-number state by $\Delta t$, and they let us integrate many initial conditions at once without writing any new loops.

Because we're doing mechanics, from now on we'll call the independent variable $t$ (time) and use $x$ for position.

# Second-Order ODEs Are Two First-Order ODEs

A second-order ODE gives the second derivative in terms of the time, the position, and the velocity,
\begin{align}
    \frac{d^2x}{dt^2} &= a(t, x, dx/dt).
\end{align}
Newton's second law is exactly this, with $a = F/m$.

The trick is to give the velocity its own name, $v = dx/dt$.  Then the one second-order equation becomes two first-order equations,
\begin{align}
    \frac{dx}{dt} &= v
    &
    \frac{dv}{dt} &= a(t, x, v),
\end{align}
which we can write together as a *vector* of equations
\begin{align}
    \frac{d}{dt}\begin{pmatrix} x \\ v \end{pmatrix} &= \begin{pmatrix} v \\ a(t, x, v) \end{pmatrix}.
\end{align}
This is exactly the form $dy/dt = f(t, y)$ we've been studying!  The difference is that now the *state* $y = (x, v)$ is two numbers instead of one, and $f$ produces two numbers too: the rate of change of each.

If you think back to your understanding of mechanics, to predict an object's motion we always needed to provide *two* initial conditions, a starting position and a starting velocity.  Now you can see that this is an absolute necessity from the ODE point of view---in order to get started integrating an ODE you've got to state your starting point $y_0$ and now $y_0$ is a pair of numbers $(x_0, v_0)$!

## A State Made of Two Numbers

We'll store a state in a numpy array whose first entry is the position and whose second entry is the velocity.

In [ ]:
x_0 = 1
v_0 = 0
y_0 = np.array([x_0, v_0])

print(y_0.shape)

Its *shape* is `(2,)`: one axis with two entries.

Arithmetic between arrays of the same shape happens *elementwise*, entry by entry.

In [ ]:
slope = np.array([0.2, -1.0])   # dx/dt = 0.2, dv/dt = -1
Delta_t = 0.1

print(y_0)
print(slope * Delta_t)
print(y_0 + slope * Delta_t)

In the second line we multiplied an array by a single number and numpy multiplied every entry.  This is the simplest example of **broadcasting**: when you combine a scalar with an array, numpy acts as if the scalar were an array of the same shape with that value in every entry.

The last line is exactly one Euler step, $y + f\,\Delta t$, for a two-number state.  The formula didn't need to change at all!

## Second-Order ODEs as Classes

Here is our base class again, now written in terms of $t$.  (We leave out the slope field this time; with a two-number state there's no single slope to draw.)

In [ ]:
class FirstOrderODE:
    '''
    A first-order ODE specifies the rate of change of a state y(t)
    in terms of
     - the state itself, y
     - the time, t.

    The generic form for an ODE is

    dy
    --  = f(t, y)
    dt

    '''

    def f(self, t, y):
        raise NotImplementedError('The generic ODE has an unspecified slope function f.')

    def exact_solution(self, t, *args):
        raise NotImplementedError('The exact solution is not known (or at least not provided).')

A `SecondOrderODE` *is* a `FirstOrderODE` whose state is $(x, v)$.  A particular second-order ODE only needs to say what the acceleration is.  The reduction to two first-order equations is the same every time, so we write it once, in the base class.

In [ ]:
class SecondOrderODE(FirstOrderODE):
    '''
    A second-order ODE specifies the acceleration

    d^2x
    ----  = a(t, x, v)
    dt^2

    where v = dx/dt is the velocity.  Treating the state as y = (x, v) turns it into
    a first-order ODE,

    d  ( x )   (    v     )
    -- (   ) = (          )
    dt ( v )   ( a(t,x,v) )

    so a subclass only needs to provide the acceleration.
    '''

    def acceleration(self, t, x, v):
        raise NotImplementedError('The generic second-order ODE has an unspecified acceleration.')

    def f(self, t, y):
        '''
        f is the vector that tells you how the state y changes.

        This is important for understanding how to treat a
        second-order ODE as a system of first-order ODEs.
        '''
        # Unpack the state into its two components,
        x, v = y
        # and pack their rates of change back up the same way.
        return np.array([v, self.acceleration(t, x, v)])

# The Harmonic Oscillator

The *harmonic oscillator* comes up often in physics as a model of things that oscillate (go back and forth).

The simplest description is in terms of [Hooke's spring law](https://en.wikipedia.org/wiki/Hooke%27s_law), (*ut tensio, sic vis*, or *as the extension, so the force*),
\begin{align}
    F = - k x
\end{align}
which says that the force is linearly proportional to the displacement from equilibrium (at $x=0$) and the slope is controlled by *the spring constant* $k$ which summarizes the material properties, geometry, and other aspects of the spring.

If we fix one end of a horizontal spring to a wall and attach a mass $m$ to the other then we can use Newton's second law $F=ma$, giving
\begin{align}
    m a &= - k x
    &
    a &= - \frac{k}{m} x
\end{align}
and we see that the *spring constant* $k$ and the mass $m$ really show up together in this equation, and we rename the combination $\omega^2 = k/m$.

So, the *harmonic oscillator* is described by the *equation of motion* $a = -\omega^2 x$.

The harmonic oscillator is extremely precious to us because it has an exact solution,
\begin{align}
    x(t) &= x_0 \cos \omega t + \frac{v_0}{\omega} \sin \omega t
    &
    v(t) &= -x_0\, \omega \sin \omega t + v_0 \cos \omega t.
\end{align}
You can check that if you plug in $t=0$ to $x(t)$ you get $x_0$ and if you plug $t=0$ into $v(t)$ you get $v_0$.

The total mechanical energy $\mathcal{E}$ is the sum of the kinetic energy $K=\frac{1}{2} m v^2$ and the potential energy in the spring $V=\frac{1}{2} k x^2$, and it is conserved because there are no frictional forces involved.
We can also think about the energy per unit mass, $E=\mathcal{E}/m$,
\begin{align}
    E &= \frac{1}{2} v^2 + \frac{1}{2} \omega^2 x^2.
\end{align}
which is also conserved.  We'll just call this 'the energy' rather than 'the energy per unit mass'.
You can check the energy is conserved: plug in the exact solution and you'll find that the energy *does not depend on time*.

In [ ]:
class HarmonicOscillator(SecondOrderODE):
    '''
    A harmonic oscillator has acceleration a = -ω^2 x.
    '''

    def __init__(self, omega):
        '''
        omega is the angular frequency, specified by ω^2 = k/m.
        '''
        self.omega = omega

    def acceleration(self, t, x, v):
        return -self.omega**2 * x

    def exact_solution(self, t, x_0, v_0):
        x =  x_0 * np.cos(self.omega * t) + v_0 / self.omega * np.sin(self.omega * t)
        v = -x_0 * self.omega * np.sin(self.omega * t) + v_0 * np.cos(self.omega * t)
        return np.array([x, v])

    def energy(self, x, v):
        # The energy per unit mass
        return 0.5 * v**2 + 0.5 * self.omega**2 * x**2

Let's imagine we pull the mass to the right $(x_0 = 1)$ and release from rest $(v_0 = 0)$,

In [ ]:
x_0, v_0 = 1., 0.
y_0 = np.array([x_0, v_0])

omega = 1.0
oscillator = HarmonicOscillator(omega)

dydt = oscillator.f(0, y_0)
dxdt, dvdt = dydt
print(dxdt, dvdt)

So, at $x=1$, $v=0$ the position isn't changing yet ($dx/dt=0$) but the velocity is decreasing ($dv/dt = -1$), meaning that the mass is starting to accelerate to the left.  That makes physical sense, given that we pulled it to the right!

## Integrating a Two-Number State

Our integrators never ask what *kind* of ODE they were given.  They only ever call its `.f` method.  Any object with an `f` works, whether or not it inherits from `FirstOrderODE`.  This is called *duck typing* ("if it walks like a duck and talks like a duck, it IS a duck!").

In [ ]:
class FirstOrderODEIntegrator:
    '''
    An integrator is an algorithm for producing numerical solutions to ODEs.
    '''

    def __init__(self, ODE):

        self.ODE = ODE

    def starting_from(self, t_0, y_0):
        raise NotImplementedError('No integration algorithm has been specified.')

So Euler's method should just work on a `SecondOrderODE`.  Almost!  One line breaks.  Our Euler method stored the solution with
```python
y = np.zeros_like(t, dtype=float)
# (previously we actually had the independent variable x, not t)
```
which makes room for *one number* at each time.  Trying to put a whole state into that slot with `y[0] = y_0` raises
```
ValueError: setting an array element with a sequence.
```

We need room for a whole state at every time.  `np.shape(y_0)` is the shape of one state,

In [ ]:
print(np.shape(y_0))

so we can ask for storage of shape `(steps+1,) + np.shape(y_0)`.  Adding tuples glues them together: for our state that's `(steps+1,) + (2,) = (steps+1, 2)`.  For a single number `np.shape` gives the empty tuple `()`, so the old first-order ODEs still work too.

In [ ]:
just_a_number = 7.1
print(np.shape(just_a_number))

In [ ]:
class EulerMethod(FirstOrderODEIntegrator):
    '''
    Euler's method is based on the Taylor series approximation

    y(t + Δt) = y(t) + y'(t) Δt + (higher-order terms with more powers of Δt which we neglect)

    Given a starting location on the solution (t_0, y_0) we can repeatedly say
    - the next t will be t_0 + Δt
    - the next y will be y_0 + Δy, and using the Taylor series Δy = y'(t) Δt

    This produces the next point on the solution, and we can begin again.
    '''

    def __init__(self, ODE, Delta_t, steps):

        self.Delta_t = Delta_t
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, t_0, y_0):

        # Populate all the t values
        t = t_0 + self.Delta_t * np.arange(self.steps+1)
        # and allocate an array to store the y values we compute.
        # 
        # Previously we had
        #
        # y = np.zeros_like(t, dtype=float)
        #
        # but this only has room for one number at each time.
        y = np.zeros((self.steps+1,) + np.shape(y_0))   # room for a whole state at each time

        # The user picked a starting y value
        y[0] = y_0

        # and we can step through the Euler procedure over and over again.
        for step in range(self.steps):
            slope = self.ODE.f(t[step], y[step])
            y[step+1] = y[step] + slope * self.Delta_t

        return t, y

That's the only change we need to make!  Let's see!

In [ ]:
euler = EulerMethod(oscillator, 0.01, 1000)
t, y = euler.starting_from(0, y_0)
print(y.shape)

The solution has shape `(1001, 2)`: one row for every time, and in each row a whole state.  So
 - `y[i]` is the state at step `i`,
 - `y[:, 0]` is the position at every time, and
 - `y[:, 1]` is the velocity at every time.

The exact solution was computed all at once from the whole array of times, so its shape is `(2,)` + the shape of `t`: components come first, and `exact[0]` is the position at every time.

In [ ]:
exact = oscillator.exact_solution(t, 1.0, 0.0)
print(exact.shape)

fig, ax = plt.subplots()
ax.plot(t, y[:, 0], label='Euler, Δt=0.01')
ax.plot(t, exact[0], label='exact', color='black', linestyle='dashed')
ax.set_xlabel('t')
ax.set_ylabel('x')
ax.legend()

That looks good, although if you look closely Euler's oscillations are slowly growing.

Let's integrate further in time to see if it's a systematic issue.

In [ ]:
euler_long = EulerMethod(oscillator, 0.01, 10000)
t_long, y_long = euler_long.starting_from(0, y_0)
print(y_long.shape)

exact = oscillator.exact_solution(t_long, 1.0, 0.0)
print(exact.shape)

fig, ax = plt.subplots(2, sharex='col')
ax[0].plot(t_long, y_long[:, 0], label='Euler, Δt=0.01')
ax[0].plot(t_long, exact[0], label='exact', color='black', linestyle='dashed')
ax[0].set_xlabel('t')
ax[0].set_ylabel('x')
ax[0].legend()

ax[1].plot(t_long, oscillator.energy(*y_long.T) / oscillator.energy(*y_0))
ax[1].set_xlabel('t')
ax[1].set_ylabel('E(t) / E(0)')

You can see that every swing of the mass grows in size.  That means the energy is increasing every swing, as shown in the bottom panel, and that's obviously unphysical.  

Hold that thought.  It's an important observation and we'll come back to it at the end.

# Many Initial Conditions at Once

Often we want many solutions, for example the oscillator released from several different positions.  We could loop over the initial conditions, calling `starting_from` each time.  Instead, let's make the state itself hold many states.

We'll keep the components first: row 0 holds all the positions and row 1 holds all the velocities.

In [ ]:
x_0_batch = np.linspace(0.2, 1.0, 5)
v_0_batch = np.zeros_like(x_0_batch)
y_0_batch = np.array([x_0_batch, v_0_batch])

print(y_0_batch.shape)
print(y_0_batch)

Does our `f` still work?  Follow it through `SecondOrderODE.f`:
 - `x, v = y` unpacks along the *first* axis, the component axis.  So `x` and `v` each have shape `(5,)`: all five positions and all five velocities.
 - The acceleration `-self.omega**2 * x` is a scalar times an array, so it has shape `(5,)` too.
 - `np.array([v, a])` packs them back up into shape `(2, 5)`.

In [ ]:
print(oscillator.f(0, y_0_batch).shape)

This is the promise `f` must keep: if the state has shape `(2,) + B`, where `B` is the *batch shape* of however many states we have, then `f` must return the same shape `(2,) + B`.  For a single state `B = ()`; for our five states `B = (5,)`.

We can check the promise by comparing with computing each state separately (`y_0_batch[:, i]` is state number `i`).

In [ ]:
batched = oscillator.f(0, y_0_batch)
one_at_a_time = np.array([oscillator.f(0, y_0_batch[:, i]) for i in range(5)])

# one_at_a_time lists the states first, shape (5, 2); .T swaps the axes to (2, 5).
print(np.allclose(batched, one_at_a_time.T))

The integrator needs no change either.  Now `np.shape(y_0_batch)` is `(2, 5)`, so the storage has shape `(steps+1, 2, 5)`, and each Euler step `y[step] + slope * Delta_t` combines arrays of shape `(2, 5)`.

In [ ]:
t, y_batch = euler.starting_from(0, y_0_batch)
print(y_batch.shape)

`y_batch[:, 0]` has shape `(1001, 5)`: the position of every state at every time.  When `ax.plot` receives a two-dimensional array it draws each column as its own curve, so one call draws all five solutions.

To compare with the exact solution we need it for every time *and* every initial condition.  But...

In [ ]:
try:
    exact = oscillator.exact_solution(t, x_0_batch, v_0_batch)
except ValueError as error:
    print(error)

## The Broadcasting Rules

We've combined arrays with scalars and arrays with arrays of the same shape.  What about two arrays of *different* shapes, like `t` with shape `(1001,)` and `x_0_batch` with shape `(5,)`?  Numpy's rules are:

1. Line the shapes up **from the right** (starting with the last axis).
2. Two axes are compatible if they're **equal**, or if one of them is **1**.  An axis of length 1 gets stretched to match the other.
3. If one shape has fewer axes, pretend it has extra axes of length 1 **on the left**.

The result has the larger length along every axis.

| shape A     | shape B  | result        |                                          |
|-------------|----------|---------------|------------------------------------------|
| `(5,)`      | `()`     | `(5,)`        | a scalar broadcasts against anything     |
| `(2, 5)`    | `(5,)`   | `(2, 5)`      | `(5,)` is treated like `(1, 5)`          |
| `(1001, 1)` | `(5,)`   | `(1001, 5)`   | the 1 stretches to 5, the missing axis to 1001 |
| `(1001,)`   | `(5,)`   | error         | 1001 ≠ 5 and neither is 1                |
| `(2, 5)`    | `(2,)`   | error         | lined up from the right, 5 ≠ 2           |

The last row surprises people: numpy lines shapes up from the right, not the left.

We want an answer for every (time, initial condition) pair, which is shape `(1001, 5)`.  We can give `t` an extra axis of length 1 with `np.newaxis`: `t[:, np.newaxis]` has shape `(1001, 1)`, which the third row says broadcasts against `(5,)` to give `(1001, 5)`.

In [ ]:
print(t[:, np.newaxis].shape)

exact = oscillator.exact_solution(t[:, np.newaxis], x_0_batch, v_0_batch)
print(exact.shape)

Components come first again, so `exact[0]` has shape `(1001, 5)` just like `y_batch[:, 0]`.

In [ ]:
fig, ax = plt.subplots()
ax.plot(t, y_batch[:, 0])
ax.plot(t, exact[0], color='black', linestyle='dashed', alpha=0.5)
ax.set_xlabel('t')
ax.set_ylabel('x')

## When `f` Doesn't Broadcast

Our harmonic oscillator kept its promise automatically because its acceleration only used elementwise arithmetic.  Here's an easy way to break it: free fall, where the acceleration $a = -g$ is a constant.

In [ ]:
class FreeFall(SecondOrderODE):
    '''
    Free fall near the Earth's surface has constant acceleration a = -g.
    '''

    def __init__(self, g):
        self.g = g

    def acceleration(self, t, x, v):
        return -self.g   # one number, no matter how many states there are!

falling = FreeFall(9.8)

# One state is fine,
print(falling.f(0, np.array([10.0, 0.0])))

# but five states are not.
try:
    print(falling.f(0, y_0_batch))
except ValueError as error:
    print(error)

With five states `v` has shape `(5,)` but the acceleration is a single number, so `np.array([v, a])` can't make a rectangular `(2, 5)` array.  The acceleration needs one entry for every state: the same shape as `x`.  `np.full_like(x, value)` makes an array shaped like `x` with `value` in every entry.

In [ ]:
class FreeFall(SecondOrderODE):
    '''
    Free fall near the Earth's surface has constant acceleration a = -g.
    '''

    def __init__(self, g):
        self.g = g

    def acceleration(self, t, x, v):
        # The same acceleration for every state in the batch
        return np.full_like(x, -self.g)

falling = FreeFall(9.8)
print(falling.f(0, np.array([10.0, 0.0])))
print(falling.f(0, y_0_batch))

Two gotchas that you might have to worry about:
 - not every library can operate on whole `numpy` arrays at once.  For example the `math` module only operates on single numbers at a time.
 - You have to think about some control structures differently.  For example, `if x > 5:` needs a single `True` or `False`, but with many states `x > 5` is an array of them.  You can use `np.where` instead.

In [ ]:
x = np.arange(10)
print(x)
print(x>5)

try:
    if x > 5:
        x = 2 * x
except:
    print('The normal if statement fails.')

x = np.where(x>5, 2*x, x)
print(x)

# The Pendulum

An ideal pendulum is a *bob* of mass $m$ on a rigid massless rod of length $\ell$ that hangs from and swings about a pivot.  The equilibrium configuration is when the bob hangs straight down below the pivot.  But we can raise the bob so that the rod makes an angle $\theta$ with the vertical, as shown in the figure below.

Two forces act on the bob: its weight $mg$, straight down, and the tension in the rod, pointing back toward the pivot.

In [ ]:
# A free-body diagram for the simple pendulum, drawn at one moment of its swing.
theta = np.radians(35)
ell = 1.0
weight = 0.45   # the length of the arrow representing mg

bob = ell * np.array([np.sin(theta), -np.cos(theta)])
outward = bob / ell                                  # unit vector pointing from the pivot out to the bob
forward = np.array([np.cos(theta), np.sin(theta)])   # unit vector along the swing, toward increasing θ

fig, ax = plt.subplots(figsize=(6, 6))

def arrow(start, vector, color, linestyle='solid'):
    ax.annotate('', xy=start+vector, xytext=start,
                arrowprops=dict(arrowstyle='-|>', color=color, linewidth=2, linestyle=linestyle, shrinkA=0, shrinkB=0))

# The ceiling, the vertical, the rod, and the path the bob swings along
ax.plot([-0.3, 0.3], [0, 0], color='black', linewidth=4)
ax.plot([0, 0], [0, -1.1*ell], color='gray', linestyle='dashed')
ax.plot([0, bob[0]], [0, bob[1]], color='black')
swing = np.radians(np.linspace(-50, 50, 100))
ax.plot(ell*np.sin(swing), -ell*np.cos(swing), color='gray', linestyle='dotted')

# The angle θ and the length ℓ
arc = np.linspace(0, theta, 30)
ax.plot(0.3*np.sin(arc), -0.3*np.cos(arc), color='black')
ax.text(0.05, -0.42, r'$\theta$', fontsize=18)
ax.text(*(0.3*bob + np.array([0.06, 0.03])), r'$\ell$', fontsize=18)

# The bob
ax.plot(*bob, marker='o', markersize=24, color='C0', zorder=3)
ax.text(*(bob + np.array([0.06, 0.04])), r'$m$', fontsize=18)

# The two forces: gravity straight down, and the tension in the rod pointing back toward the pivot
arrow(bob, weight * np.array([0, -1]), 'C3')
ax.text(*(bob + np.array([0.02, -weight - 0.08])), r'$mg$', fontsize=18, color='C3')
arrow(bob, -0.4 * outward, 'C2')
ax.text(*(bob - 0.4*outward + 0.1*forward), r'Tension', fontsize=18, color='C2')

# Gravity broken into a piece along the rod and a piece along the swing
arrow(bob, weight * np.cos(theta) * outward, 'C3', 'dashed')
ax.text(*(bob + weight*np.cos(theta)*outward + np.array([0.03, -0.02])), r'$mg\cos\theta$', fontsize=16, color='C3')
arrow(bob, -weight * np.sin(theta) * forward, 'C3', 'dashed')
ax.text(*(bob - weight*np.sin(theta)*forward + np.array([-0.55, -0.1])), r'$mg\sin\theta$', fontsize=16, color='C3')

# Fix the limits by hand: matplotlib skips any arrow whose tip is outside the axes.
ax.set_xlim(-0.75, 1.2)
ax.set_ylim(-1.35, 0.1)
ax.set_aspect('equal')
ax.axis('off')

The tension and the radial component of the gravitational force $mg\cos\theta$ keep the bob on the circle and don't contribute to the bob's motion along the circle.

But the gravitational force also has a tangential component, $mg\sin\theta$.

The rod keeps the bob on the circle of radius $\ell$, so the only way it can move is along that circle (the dotted arc).  Its distance along the arc is $s = \ell\theta$, so its acceleration along the arc is $\frac{d^2s}{dt^2} = \ell\frac{d^2\theta}{dt^2}$.

The tangential component of Newton's second law along the arc is therefore
\begin{align}
    m \ell\, \frac{d^2\theta}{dt^2} &=- m g \sin\theta.
\end{align}

Dividing by $m\ell$ (the mass cancels!), a simple pendulum obeys
\begin{align}
    \frac{d^2\theta}{dt^2} &= -\omega^2 \sin \theta
    &
    \omega^2 &= g/\ell.
\end{align}
For small swings $\sin\theta \approx \theta$ and this is the harmonic oscillator, with period $2\pi/\omega$ no matter the amplitude.  But for the full equation of motion there is no closed-form solution.  So this is a good target for numerical integration!

We'll use `x` for the angle $\theta$ and `v` for the angular velocity $d\theta/dt$.

In [ ]:
class Pendulum(SecondOrderODE):
    '''
    A simple pendulum has angular acceleration a = -ω^2 sin(θ), where ω^2 = g/ℓ.
    We use x for the angle θ and v for the angular velocity.
    '''

    def __init__(self, omega):
        self.omega = omega

    def acceleration(self, t, x, v):
        return -self.omega**2 * np.sin(x)

Let's release the pendulum from rest at several amplitudes, all at once.

In [ ]:
pendulum = Pendulum(1.0)

theta_0 = np.array([0.1, 0.5, 1.0, 2.0, 3.0])
rest = np.zeros_like(theta_0)
y_0 = np.array([theta_0, rest])


euler = EulerMethod(pendulum, 0.001, 20000)
t, y = euler.starting_from(0, y_0)
print(y.shape)

To compare the *shapes* of the swings let's divide each angle by its amplitude.  That's more broadcasting: `y[:, 0]` has shape `(20001, 5)` and `theta_0` has shape `(5,)`, so each column gets divided by its own amplitude.

In [ ]:
fig, ax = plt.subplots()
ax.plot(t, y[:, 0] / theta_0)
ax.set_xlabel('t')
ax.set_ylabel('θ / θ_0')
ax.legend([f'θ_0 = {amplitude}' for amplitude in theta_0])

Let's watch them swing, side by side and in real time.  For the movie's clock to tick in seconds, $t$ needs units, and so does $\omega$.  We'll use a 1-meter rod on Earth, so $\omega = \sqrt{g/\ell} \approx 3.13$ per second and small swings take about $2\pi/\omega \approx 2$ seconds.

The black arrow on each bob is the tangential piece of gravity, the force that drives the swing.  It vanishes at the bottom and is biggest when the rod is horizontal.

You may need to run `uv sync` in a terminal from the root directory of this repository for this to work.

In [ ]:
import imageio_ffmpeg
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Point matplotlib at the copy of ffmpeg that came with `uv sync`.
plt.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

# A real pendulum: a 1-meter rod on Earth.
g = 9.8      # m/s^2
ell = 1.0    # m
lab_pendulum = Pendulum(np.sqrt(g / ell))

# Ten seconds of motion for each amplitude.  With ω^2 ≈ 9.8 / s^2 we need a small step to keep Euler's energy drift
# under control (it grows like exp(ω^2 Δt t_f), about 1% here).
Delta_t = 0.0001
t, y = EulerMethod(lab_pendulum, Delta_t, 100000).starting_from(0, np.array([theta_0, np.zeros_like(theta_0)]))

# Show every frame_step-th step, and play one frame every frame_step * Delta_t seconds, so the movie runs in real time.
frame_step = 400                                       # 400 * 0.0001 s = 0.04 s per frame, 25 frames per second
frames = range(0, len(t), frame_step)

# Where each pendulum hangs from, side by side
pivot_x = 2.5 * np.arange(len(theta_0))

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(pivot_x, np.zeros_like(pivot_x), marker='o', color='black', linestyle='none')
for x, amplitude in zip(pivot_x, theta_0):
    ax.text(x, -1.45, f'θ_0 = {amplitude}', ha='center')

rods = [ax.plot([], [], color='black')[0] for _ in theta_0]
# One color per pendulum, matching its trace in the θ / θ_0 figure above
colors = [f'C{i}' for i in range(len(theta_0))]
bobs = ax.scatter(pivot_x, np.zeros_like(pivot_x), s=16**2, color=colors, zorder=3)
# The tangential piece of gravity, the force that drives the swing
force = ax.quiver(pivot_x, np.zeros_like(pivot_x), np.zeros_like(pivot_x), np.zeros_like(pivot_x),
                  color='black', angles='xy', scale_units='xy', scale=1.5*g,   # a force of g is drawn 1/1.5 m long
                  width=0.003, headwidth=4, headlength=5, minlength=0, zorder=4)
clock = ax.set_title('')

ax.set_xlim(pivot_x[0] - 1.3, pivot_x[-1] + 1.3)
ax.set_ylim(-1.6, 1.3)
ax.set_aspect('equal')
ax.axis('off')

def draw(step):
    theta = y[step, 0]                                 # all five angles at this time
    bob_x = pivot_x + ell * np.sin(theta)
    bob_y =         - ell * np.cos(theta)

    for rod, x, bx, by in zip(rods, pivot_x, bob_x, bob_y):
        rod.set_data([x, bx], [0, by])
    bobs.set_offsets(np.column_stack([bob_x, bob_y]))

    # Per unit mass the tangential force is g sin θ, pointing back toward the bottom of the swing.
    tangential = -g * np.sin(theta)
    force.set_offsets(np.column_stack([bob_x, bob_y]))
    force.set_UVC(tangential * np.cos(theta), tangential * np.sin(theta))

    clock.set_text(f't = {t[step]:.2f} s')

animation = FuncAnimation(fig, draw, frames=frames, interval=1000 * frame_step * Delta_t)
plt.close(fig)   # show only the movie, not a leftover still frame
HTML(animation.to_html5_video())

Big swings take longer, and near $\theta_0 = \pi$ (balanced upside-down) the pendulum lingers at the top.

We can measure the period of every pendulum at once.  Each starts at rest at $\theta_0$ and swings with negative velocity until it comes to rest at $-\theta_0$, half a period later.  `y[1:, 1] >= 0` is an array of `True`s and `False`s with shape `(steps, simulations)`, and `np.argmax(..., axis=0)` finds the index of the first `True` in each column.

Since only the array size changes, let's use many more amplitudes.

In [ ]:
theta_0 = np.linspace(0.1, 3.0, 30)
rest = np.zeros_like(theta_0)
y_0 = np.array([theta_0, rest])
t, y = euler.starting_from(0, y_0)

# The first step (after the start) at which each pendulum's velocity is no longer negative
half_period_step = 1 + np.argmax(y[1:, 1] >= 0, axis=0)
period = 2 * t[half_period_step]

[The period actually *is* known in closed form](https://en.wikipedia.org/wiki/Pendulum_(mechanics)#Arbitrary-amplitude_period), in terms of a special function called the complete elliptic integral $K$,
\begin{align}
    T &= \frac{4}{\omega} K\left(\sin^2 \frac{\theta_0}{2}\right),
\end{align}
which lets us check our measurement.

In [ ]:
from scipy.special import ellipk

amplitude = np.linspace(0, 3.1, 200)

fig, ax = plt.subplots()
ax.plot(amplitude, 4 / pendulum.omega * ellipk(np.sin(amplitude/2)**2), color='black', label='exact')
ax.plot(theta_0, period, marker='o', linestyle='none', label='Euler, Δt=0.001')
ax.axhline(2*np.pi / pendulum.omega, color='gray', linestyle='dotted', label='small-angle 2π/ω')
ax.set_xlabel('amplitude θ_0')
ax.set_ylabel('period')
ax.legend()

# Energy Isn't Conserved

It seems as though all of our expertise we built up studying first-order ODEs is enough to serve us well in our physics simulations.  However, neither the Euler method nor RK2 is typically used for accurate simulations.

The issue is that these integrators do not respect *energy conservation*.

Reconsider the harmonic oscillator.  Its energy $E = \frac{1}{2}v^2 + \frac{1}{2}\omega^2 x^2$ should be constant, so in *phase space*, the $(x, v)$ plane, the exact solution goes around the same ellipse forever (a circle when $\omega = 1$).

Let's use a big step, $\Delta t = 0.1$, so it's easy to see what Euler's method does.

In [ ]:
oscillator = HarmonicOscillator(1.0)
euler = EulerMethod(oscillator, 0.1, 100)
t, y = euler.starting_from(0, np.array([1.0, 0.0]))
exact = oscillator.exact_solution(t, 1.0, 0.0)

fig, ax = plt.subplots()
ax.plot(y[:, 0], y[:, 1], label='Euler, Δt=0.1')
ax.plot(exact[0], exact[1], color='black', linestyle='dashed', label='exact')
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('v')
ax.legend()

Euler's method spirals outward: every orbit has more energy than the last.

We can see exactly how much.  One Euler step takes $x \to x + v\,\Delta t$ and $v \to v - \omega^2 x\,\Delta t$, so the new energy is
\begin{align}
    \frac{1}{2}\left(v - \omega^2 x \Delta t\right)^2 + \frac{1}{2}\omega^2 \left(x + v \Delta t\right)^2
    &= \frac{1}{2}v^2 + \frac{1}{2}\omega^2 x^2 + \frac{1}{2}\omega^2 \Delta t^2 \left(v^2 + \omega^2 x^2\right)
    = E \left(1 + \omega^2 \Delta t^2\right).
\end{align}
The cross terms cancel, and *every single step* multiplies the energy by $1 + \omega^2 \Delta t^2 > 1$.

Making $\Delta t$ smaller brings that factor closer to 1, but reaching a time $t_f$ takes more steps, $t_f/\Delta t$ of them, so
\begin{align}
    \frac{E(t_f)}{E(0)} &= \left(1 + \omega^2 \Delta t^2\right)^{t_f/\Delta t} \approx e^{\omega^2 \Delta t\, t_f}.
\end{align}
A smaller $\Delta t$ only slows the growth down.  The energy still grows exponentially, and if you integrate long enough it blows up.

## Does a Higher-Order Method Help?

Here's our second-order Runge-Kutta method with the same one-line storage fix.

In [ ]:
class RungeKutta2(FirstOrderODEIntegrator):

    def __init__(self, ODE, alpha, Delta_t, steps):

        # Conditions for matching the Taylor expansion to 2nd order
        self.alpha = alpha
        self.b = 1/(2*self.alpha)
        self.a = 1 - self.b

        self.Delta_t = Delta_t
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, t_0, y_0):
        # Populate all the t values
        t = t_0 + self.Delta_t * np.arange(self.steps+1)
        # and allocate an array to store the y values we compute.
        # y = np.zeros_like(t, dtype=float)   # room for only one number at each time
        y = np.zeros((self.steps+1,) + np.shape(y_0))   # room for a whole state at each time

        # The user picked a starting y value
        y[0] = y_0

        # and we can step through the RK2 procedure over and over again.
        for step in range(self.steps):
            k_1 = self.ODE.f(t[step], y[step])
            k_2 = self.ODE.f(t[step] + self.alpha * self.Delta_t, y[step] + self.alpha * k_1 * self.Delta_t)

            y[step+1] = y[step] + (self.a * k_1 + self.b * k_2) * self.Delta_t

        return t, y

Let's integrate for a long time, to $t_f = 1000$ (about 160 periods), and track the energy.  `oscillator.energy(y[:, 0], y[:, 1])` hands the energy every position and every velocity at once, so it computes the energy at every time.

In [ ]:
Delta_t = 0.1
steps = 10000

t, euler_y = EulerMethod(oscillator, Delta_t, steps).starting_from(0, np.array([1.0, 0.0]))
t, rk2_y   = RungeKutta2(oscillator, 0.5, Delta_t, steps).starting_from(0, np.array([1.0, 0.0]))

euler_energy = oscillator.energy(euler_y[:, 0], euler_y[:, 1])
rk2_energy   = oscillator.energy(rk2_y[:, 0],   rk2_y[:, 1])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(t, euler_energy / euler_energy[0], color='C0', label=f'EulerMethod, Δt={Delta_t}')
axes[0].plot(t, rk2_energy / rk2_energy[0], color='C1', label=f'RungeKutta2, Δt={Delta_t}')
axes[0].set_yscale('log')

# Zoom in on RK2 alone, where the log scale hides the drift
axes[1].plot(t, rk2_energy / rk2_energy[0], color='C1', label=f'RungeKutta2, Δt={Delta_t}')

for axis in axes:
    axis.set_xlabel('t')
    axis.set_ylabel('E(t) / E(0)')
    axis.legend()

The phase-space picture tells the same story.  On the left, over the first ten time units, Euler's method spirals visibly outward while RK2 sits right on the exact circle.  On the right we follow RK2 for the whole run to $t=1000$, colored by time.  It spirals outward too, just so slowly that each orbit nearly lands on the last, and together they fill in a band instead of tracing a single circle.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex='row', sharey='row')

# Left: the same two runs, but only the first ten time units
early = round(10 / Delta_t) + 1   # the number of steps with t ≤ 10
exact = oscillator.exact_solution(t[:early], 1.0, 0.0)

axes[0].plot(euler_y[:early, 0], euler_y[:early, 1], color='C0', label=f'EulerMethod, Δt={Delta_t}')
axes[0].plot(rk2_y[:early, 0],   rk2_y[:early, 1],   color='C1', label=f'RungeKutta2, Δt={Delta_t}')
axes[0].plot(exact[0], exact[1], color='black', linestyle='dashed', label='exact')
axes[0].set_title(f't ≤ {t[early-1]:.0f}')
axes[0].legend()

# Right: RK2 for the whole long run, colored by time
points = axes[1].scatter(rk2_y[:, 0], rk2_y[:, 1], c=t, s=1)
fig.colorbar(points, ax=axes, label='t')
axes[1].set_title(f'RungeKutta2, Δt={Delta_t}, t ≤ {t[-1]:.0f}')

for axis in axes:
    axis.set_aspect('equal')
    axis.set_xlabel('x')
    axis.set_ylabel('v')

RK2 is *enormously* better.  By $t=1000$ Euler's energy has grown by a factor of about $10^{43}$, while RK2's has grown by less than 30%.

But look closely: RK2's energy is still climbing.  For the harmonic oscillator each RK2 step multiplies the energy by $1 + \frac{1}{4}\omega^4 \Delta t^4$.  That's much closer to 1 than Euler's factor, but it's still bigger than 1, and it compounds step after step.  A higher-order method makes the leak smaller, but it doesn't plug it.

This is a *physics* problem, not just a numerical one.  Energy conservation is one of the most important facts about this system, and neither integrator gets it right.  Every step nudges the energy the same way, and the nudges add up.  In a subsequent lesson we'll build integrators that respect this structure, whose energy errors stay bounded no matter how long we integrate.